In [ ]:
import nltk
import spacy
import duckdb
import rapidfuzz
from transformers import BertTokenizerFast, Trainer
import torch

# Detecting Idioms in Sentence

To replace idioms in a sentence, we first have to ***detect*** and ***locate*** idioms.

There are two approaches to this:

1. Fuzzy matching

2. BERT (Bidirectional Encoder Representation from Transformers)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("abdallahashrafx/Bert_idiom_classifier")

In [175]:
from transformers import BertModel
import torch

model = BertModel.from_pretrained("abdallahashrafx/Bert_idiom_classifier", num_labels=3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17821.43it/s]


In [ ]:
from nltk.corpus import brown
import spacy
from idiom_parser import *

nlp = spacy.load("en_core_web_sm")

corpus_raw = brown.raw("ca01")
doc = nlp(corpus_raw)
corpus_tagged = [token.pos_ for token in doc]

input_sent = "Daredevil is not above the law"
doc = nlp(input_sent)
input_tagged = [token.pos_ for token in doc]

idiomParser = Idioms()

matches = idiomParser.find_idiom_matches(input_sent)
print(matches)

In [209]:
# label2id = {"0": 0, "B-IDIOM": 1, "I-IDIOM": 2}
# id2label = {0: "0", 1: "B-IDIOM", 2: "I-IDIOM"}

id2class = {0: "has idiom", 1:"does not have idiom"}
def labelSentence(sent: str):
    if (not sent):
        return

    inputs = tokenizer(sent, return_tensors="pt", truncation=True)

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    output = model(input_ids, attention_mask)

    pooled_output = output.pooler_output

    dropped_output = torch.nn.Dropout(p=0.4)(pooled_output)

    final_output = torch.nn.Linear(model.config.hidden_size, 1)(dropped_output)

    probabilities = torch.sigmoid(final_output)

    prediction = (probabilities > 0.5).int()

    print(f"Sentence: {sent}")
    print(f"Prediction: {id2class[prediction.item()]}")


    # print(probs)
    # predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # prediction_class_id = torch.argmax(predictions, dim=-1)
    # print(prediction_class_id)

    # print(id2class[prediction_class_id.item()])

    # prediction_ids = outputs.logits.argmax(-1)[0]

    # # tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # labels = [id2label[id] for id in prediction_ids.numpy()]
    # print(labels)

In [216]:
sentence = "a piece of cake."
labelSentence(sentence)

Sentence: a piece of cake.
Prediction: does not have idiom


## How well does this perform? Evaluation Metrics

In [58]:
DATASET_PATH = "../Data/idiom_repository_all.parquet"

In [92]:
query = f"""
    SELECT
        idiom, UNNEST(LIST(usages)) AS usages
    FROM
        '{DATASET_PATH}'
    WHERE
        usages != []
    GROUP BY
        idiom
"""

res = duckdb.query(query).df()

In [93]:
res

,idiom,usages
0,12-ounce curls,"[""don't burn out your biceps with 12-ounce cur..."
1,Buggins's turn,"[""he will be appointed on the principle of bug..."
2,German goiter,"[""brothers gribble, berger, wolf, shadwill and..."
3,I can tell you,"[""we had trouble getting that grand piano up t..."
4,"Tom, Dick and Harry","[""we want the place to be accessible to any to..."
...,...,...
5129,with knobs on,"[""and the same to you with knobs on!"",""the doo..."
5130,work one's magic,"[""the company accountants worked their magic a..."
5131,work the room,"[""in show business parlance, miss hudson knows..."
5132,wrap in the flag,"[""both parties wrap themselves in the flag eve..."
